# Entanglement-level statistics: nonequilibrium versus disordered equilibrium
Compare $N_x=20,N_y=32$, hard domain walls, $\alpha_1=1$, $\alpha_2=30$, $n_{\mathrm{shell}}=1$, with 100 pure-state realizations per ensemble. The nonequilibrium states are cycle-64 endpoints. The equilibrium states are half-filled ground states with independent zero-mean Gaussian potentials on diagonal entries $(x,y,\mu)$ only on $x=5,15$, zero elsewhere, of variance $W^2=1,2,3,4,6,9,12,16,25$. Use the same half-system $A_y=16$, fixed origin $y_0=0$, both orbitals, and $|\lambda|\le0.99$.

Within EACH realization, sort $\varepsilon=-2\operatorname{artanh}\lambda$ and compute
$$\delta_n=\varepsilon_{n+1}-\varepsilon_n,\qquad r_n=\frac{\min(\delta_n,\delta_{n+1})}{\max(\delta_n,\delta_{n+1})}.$$
Only afterward pool ratios. No spacing ever joins two different realizations. Histograms and means weight each valid ratio equally. Errors on the pooled mean delete whole realizations in a jackknife. No levels or near-degenerate pairs are merged. Report unresolved double-zero spacings and gaps below $10^{-6}$ separately.

The spacing histogram uses $\delta/\overline\delta$ within each realization; this removes an overall scale but is not full local unfolding. Ratio statistics do not require that scale normalization. The analyzed object is the single-particle entanglement spectrum, with no claim of many-body chaos. The full retained spectrum is analyzed without separating wall branches; reference ensembles are diagnostic comparisons, not assumed classifications.

The clean spectrum is a separate degeneracy control. Repeat ratios at $L=0.5,0.9,0.95$ and omit triples that straddle zero as sensitivity checks. These are descriptive single-particle level statistics; no random-matrix universality class is assumed.

In [ ]:
import os
CPU_RANGE=(40,55)
selected=list(range(CPU_RANGE[0],CPU_RANGE[1]+1))
assert set(selected).issubset(os.sched_getaffinity(0))
os.sched_setaffinity(0,selected)
for key in ('OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','OMP_NUM_THREADS'):os.environ[key]=str(len(selected))
print('Allocated CPUs:',selected)

## Configuration and inputs
Validate the saved spectra by checksum; no dynamics or ground-state diagonalization is repeated.

In [ ]:
from pathlib import Path
import json,hashlib,subprocess
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display,Image
OUT=Path.cwd();BASE=OUT.parents[2]
SMALL=OUT.parent/'half_system_centered_spectrum_n20_sizes_hard_alpha1_v1'
EQ=BASE/'06_domain_wall_flattened_ground_state_reference/analysis_outputs/disordered_exact_wall_central_charge_through_ny100_v1'
L=.99
WINDOWS=[.5,.9,.95,.99]
ZERO_TOL=1e-12
PAIR_TOL=1e-6
def sha(p):
    with p.open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
inputs=[]
def verify(p,expected):
    digest=sha(p);assert digest==expected
    inputs.append({'path':str(p),'sha256':digest,'bytes':p.stat().st_size})
f=SMALL/'centered_spectra_Ny032.npz'
overlay=json.loads((SMALL/'overlay_diagnostics.json').read_text())
verify(f,overlay['sizes']['32']['spectra_sha256'])
with np.load(f) as z:
    assert np.array_equal(z['sample_ids'],np.arange(100))
    assert np.array_equal(z['subsystem_indices'],np.arange(640))
    arrays={'Nonequilibrium':z['eigenvalues']}
folder=EQ/'Ny032'
acq=json.loads((folder/'acquisition_complete.json').read_text())
provenance=json.loads((EQ/'input_provenance.json').read_text())
assert provenance['configuration']['disorder_x_columns']==[5,15]
assert acq['identity']==provenance['identity'] and acq['realizations']==900
verify(folder/'entropy_profiles.npz',acq['cache_sha256'])
for v in [1,2,3,4,6,9,12,16,25]:
    spectra=[]
    for sid in range(100):
        f=folder/'realizations'/f'W2_{v:02d}_sample_{sid:03d}.npz'
        receipt=json.loads(f.with_suffix('.json').read_text())
        assert receipt['identity']==acq['identity'] and receipt['sample_id']==sid and receipt['variance']==v
        assert receipt['bytes']==f.stat().st_size
        verify(f,receipt['sha256'])
        with np.load(f) as z:
            p=z['diagonal_potential'];mask=np.isin(np.arange(1280)//2%20,[5,15])
            assert np.all(p[~mask]==0)
            spectra.append(z['half_centered_eigenvalues'])
    arrays[f'W2={v:g}']=np.array(spectra)
f=folder/'realizations/W2_00_sample_000.npz'
r=json.loads(f.with_suffix('.json').read_text());verify(f,r['sha256'])
with np.load(f) as z:clean=z['half_centered_eigenvalues'][None,:]
for a in arrays.values():assert a.shape==(100,640)
assert clean.shape==(1,640)
labels=list(arrays)
metadata={'Nx':20,'Ny':32,'Ay':16,'origin_y':0,'samples_per_ensemble':100,
 'L':L,'windows':WINDOWS,'disorder_inward_depth':0,'disorder_x_columns':[5,15],'zero_spacing_tolerance':ZERO_TOL,'near_pair_spacing_threshold':PAIR_TOL,
 'ratio_weighting':'pool ratios after computing within realizations',
 'error_bars':'one delete-realization jackknife standard error',
 'spacing_scaling':'divide each realization spacings by their mean; not local unfolding',
 'sector_handling':'full retained single-particle spectrum; no branch separation or pair merging'}
(OUT/'input_provenance.json').write_text(json.dumps({'configuration':metadata,'inputs':inputs,
 'equilibrium_provenance':json.loads((EQ/'input_provenance.json').read_text())},indent=2)+'\n')
display(pd.Series(metadata))

## Per-realization spacings and ratios
Compute moments and histograms from the original levels. All retained energies must be finite; unresolved ratios are counted explicitly.

In [ ]:
def compute(a,window,label):
    ratio_rows=[];spacings=[];ratio_ids=[];sample_rows=[];same_sign=[]
    for sid,lam in enumerate(a):
        assert np.isfinite(lam).all() and np.max(np.abs(lam))<=1+1e-8
        selected=lam[np.abs(lam)<=window]
        e=np.sort(np.log1p(-selected)-np.log1p(selected))
        assert np.isfinite(e).all()
        d=np.diff(e)
        assert np.all(d>=0)
        if len(d):assert d.mean()>0
        den=np.maximum(d[:-1],d[1:])
        valid=den>ZERO_TOL
        r=np.minimum(d[:-1],d[1:])[valid]/den[valid]
        assert np.all((r>=0)&(r<=1))
        same=(e[:-2]*e[2:]>0)[valid]
        same_sign.extend(r[same])
        ratio_rows.extend(r);ratio_ids.extend([sid]*len(r));spacings.extend(d/d.mean() if len(d) else [])
        sample_rows.append({'ensemble':label,'L':window,'sample_id':sid,'levels':len(e),
           'ratio_count':len(r),'ratio_sum':float(r.sum()),'mean_r':float(r.mean()) if len(r) else None,
           'spacing_count':len(d),'mean_spacing':float(d.mean()) if len(d) else None,
           'near_pair_gaps':int((d<PAIR_TOL).sum()),'unresolved_ratios':int((~valid).sum())})
    r=np.array(ratio_rows);ids=np.array(ratio_ids,dtype=int)
    assert len(r)>0
    n=np.bincount(ids,minlength=len(a));s=np.bincount(ids,weights=r,minlength=len(a))
    se=None
    if len(a)>1:
        leave=(s.sum()-s)/(n.sum()-n)
        se=float(np.sqrt((len(a)-1)/len(a)*np.sum((leave-leave.mean())**2)))
    row={'ensemble':label,'L':window,'samples':len(a),'total_levels':sum(x['levels'] for x in sample_rows),
         'realizations_with_ratios':int(np.count_nonzero(n)),
         'ratio_count':len(r),'mean_r':float(r.mean()) if len(r) else None,'mean_r_jackknife_se':se,
         'same_sign_mean_r':float(np.mean(same_sign)),'same_sign_ratio_count':len(same_sign),
         'near_pair_gaps':sum(x['near_pair_gaps'] for x in sample_rows),
         'unresolved_ratios':sum(x['unresolved_ratios'] for x in sample_rows)}
    return r,np.array(spacings),ids,row,sample_rows
results={};summary_rows=[];sample_rows=[];cache={};sensitivity=[]
for i,(label,a) in enumerate(tqdm(arrays.items(),desc='Level statistics',unit='ensemble')):
    for window in WINDOWS:
        r,s,ids,row,sr=compute(a,window,label)
        sensitivity.append(row)
        if window==L:
            results[label]={'ratios':r,'spacings':s,'sample_ids':ids}
            summary_rows.append(row);sample_rows.extend(sr)
            cache[f'ratios_{i}']=r;cache[f'spacings_{i}']=s;cache[f'sample_ids_{i}']=ids
r,s,ids,clean_summary,sr=compute(clean,L,'Clean control')
cache['clean_ratios']=r;cache['clean_spacings']=s
summary=pd.DataFrame(summary_rows)
summary.to_csv(OUT/'level_statistics_summary.csv',index=False)
pd.DataFrame(sample_rows).to_csv(OUT/'sample_level_statistics.csv',index=False)
pd.DataFrame(sensitivity).to_csv(OUT/'window_sensitivity.csv',index=False)
np.savez_compressed(OUT/'level_statistics.npz',**cache,labels=labels,L=L)
display(summary[['ensemble','ratio_count','mean_r','mean_r_jackknife_se','near_pair_gaps']])
print('Clean degeneracy control:',clean_summary)

## Gap-ratio histograms
Each panel compares one equilibrium disorder strength with the same nonequilibrium reference. Both are normalized to unit area. Change R_BINS to rebin without recomputing spectra or distribution distances.

In [ ]:
strengths=np.array([float(label.split('=')[1]) for label in labels[1:]])
import matplotlib as mpl
import matplotlib.pyplot as plt
os.environ['TEXINPUTS']=str(OUT/'latex_support')+os.pathsep+os.environ.get('TEXINPUTS','')
mpl.rcParams.update({'font.family':'sans-serif','font.sans-serif':['CMU Sans Serif'],
 'font.size':10,'axes.labelsize':11,'axes.titlesize':11,'xtick.labelsize':10,'ytick.labelsize':10,
 'text.usetex':True,'text.latex.preamble':r'\usepackage{amsmath,amssymb}',
 'pdf.fonttype':42,'xtick.direction':'in','ytick.direction':'in',
 'legend.fontsize':9,'legend.frameon':False})
R_BINS=50
redges=np.linspace(0,1,R_BINS+1)
colors=['#d62728','#228833','#1565c0','#aa4499','#cc8800','#009999','#bb5566','#4477aa','#66aa55']
hist_rows=[]
density={}
for label in labels:
    n,_=np.histogram(results[label]['ratios'],redges)
    rho=n/(len(results[label]['ratios'])*np.diff(redges))
    assert n.sum()==len(results[label]['ratios']) and np.isclose(np.dot(rho,np.diff(redges)),1)
    density[label]=rho
    hist_rows.extend({'ensemble':label,'left':redges[j],'right':redges[j+1],
                      'count':int(n[j]),'density':float(rho[j])} for j in range(R_BINS))
pd.DataFrame(hist_rows).to_csv(OUT/'ratio_histograms.csv',index=False)
fig,axes=plt.subplots(3,3,figsize=(9.5,10.5),sharex=True,sharey=True)
fig.subplots_adjust(left=.075,right=.98,bottom=.05,top=.695,hspace=.32,wspace=.23)
for ax,label,var,color,letter in zip(axes.flat,labels[1:],strengths,colors,'abcdefghi'):
    rho=density[label];nz=rho>0
    ax.bar(redges[:-1][nz],rho[nz],width=np.diff(redges)[nz],align='edge',
           color=color,alpha=.55,edgecolor=color,linewidth=.3,label='Equilibrium')
    ax.stairs(density['Nonequilibrium'],redges,color='black',lw=1.1,label='Nonequilibrium')
    mean=summary.set_index('ensemble').loc[label,'mean_r']
    ax.set(title=rf'$W^2={var:g}$; $\langle R\rangle={mean:.3f}$',xlim=(0,1),ylim=(.01,60),yscale='log')
    ax.tick_params(top=True,right=True)
    ax.text(-.16,1.04,f'({letter})',transform=ax.transAxes,fontweight='bold')
axes[0,0].legend(loc='upper left')
for ax in axes[-1]:ax.set_xlabel(r'Adjacent-gap ratio $R$')
for ax in axes[:,0]:ax.set_ylabel(r'$P(R)$')
# Vector schematic: one square per unit cell, two independent orbitals per square.
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
fig.text(.045,.973,r'$R_n=\frac{\min(\delta_n,\delta_{n+1})}{\max(\delta_n,\delta_{n+1})},'
              r'\qquad \delta_n=\varepsilon_{n+1}-\varepsilon_n$',fontsize=13,va='top')
fig.text(.045,.909,r'$h_{\rm dis}=h_{\rm flat}-\mathrm{diag}(m_{x_i}v_i),'
         r'\qquad v_i\overset{\rm iid}{\sim}\mathcal{N}(0,W^2)$',fontsize=12)
fig.text(.045,.873,r'$m_x=1$ only at $x=5,15$; zero elsewhere.',fontsize=11)
fig.text(.045,.840,r'$W^2$: variance; independent potentials for each $y$ and orbital.',fontsize=10)
fig.text(.045,.813,r'Disorder extends along the full $N_y=32$; the grid shows half system $A$.',fontsize=10)
fig.text(.045,.739,r'$20\times32$; $100$ realizations each; $A_y=16$; $y_0=0$; $L=0.99$',fontsize=10)
schematic=fig.add_axes([.72,.770,.245,.18])
cells=np.zeros((16,20),dtype=int)
cells[:,5:16]=1
cells[:,[5,15]]=2
assert np.count_nonzero(cells==2)==32
schematic.pcolormesh(np.arange(21)-.5,np.arange(17)-.5,cells,
    cmap=ListedColormap(['#f0f0f0','#dceaf3','#efaa45']),vmin=0,vmax=2,
    edgecolors='#999999',linewidth=.25,shading='flat',rasterized=False)
schematic.set(aspect='equal',xlim=(-.5,19.5),ylim=(-.5,15.5),
    xticks=[0,5,10,15,19],yticks=[0,5,10,15],xlabel=r'$x$',ylabel=r'$y$')
schematic.set_title(r'Half system $A$: $20\times16$ cells',fontsize=10,pad=19)
schematic.tick_params(labelsize=8,top=False,right=False,length=2)
for wall in [5,15]:
    schematic.annotate('DW',xy=(wall,15.5),xytext=(wall,16.7),ha='center',
        fontsize=8,annotation_clip=False,
        arrowprops={'arrowstyle':'-|>','color':'black','lw':.8})
schematic.xaxis.labelpad=1;schematic.yaxis.labelpad=1
fig.legend(handles=[Patch(facecolor='#f0f0f0',edgecolor='#999999',label='Exterior'),
                    Patch(facecolor='#dceaf3',edgecolor='#999999',label='Topological slab'),
                    Patch(facecolor='#efaa45',edgecolor='#999999',label='Disorder (both orbitals)')],
           loc='upper left',bbox_to_anchor=(.04,.790),ncol=3,fontsize=9,
           handlelength=1.1,columnspacing=1.2)
fig.savefig(OUT/'gap_ratio_distributions.pdf');plt.close(fig)
subprocess.run(['pdftoppm','-r','300','-singlefile','-png',str(OUT/'gap_ratio_distributions.pdf'),str(OUT/'gap_ratio_distributions')],check=True)
display(Image(filename=str(OUT/'gap_ratio_distributions.png'),width=1100))

## Diagnostics
Spectra and input checksums are validated above. Each distribution pools ratios formed within a single realization.

In [ ]:
(OUT/'diagnostics.json').write_text(json.dumps({'configuration':metadata,'ensembles':summary_rows,'clean_control':clean_summary,'sensitivity':sensitivity,'checks':{'input_checksums':True,'within_realization_spacings_only':True,'ratio_histograms_unit_area':True}},indent=2)+'\n')
display(summary)